<a href="https://colab.research.google.com/github/simasaadi/toronto-water-analytics/blob/main/notebooks/04_export_for_tableau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 – Export for Tableau: Toronto AOC1 Water-Quality Data

This notebook prepares curated export tables from the cleaned Toronto AOC1 water-quality dataset for use in Tableau dashboards.

The previous notebooks covered:

- `01_data_cleaning.ipynb` – ingesting, cleaning, and standardizing the raw dataset  
- `02_exploration.ipynb` – high-level exploratory data analysis (EDA)  
- `03_analysis.ipynb` – advanced analysis (relationships, anomalies, temporal dynamics, spatial patterns)

Here, I focus on building **lightweight, analysis-ready tables** that can be easily consumed by Tableau Public.

## Export Objectives

1. Load the cleaned dataset (`toronto_aoc1_clean.csv`) from the GitHub repository into the Colab environment.
2. Create a **core measurement fact table** with key dimensions (time, location, characteristic).
3. Generate **aggregated tables** for:
   - monthly overall statistics  
   - monthly statistics for top characteristics  
   - spatial summaries by monitoring location  
   - seasonal medians by month  
4. Create a **trimmed fact table** with the top 1% of extreme values removed (for robust visualizations).
5. Save all exports as `.csv` files under `/content/data/tableau`, ready for download and upload to GitHub / Tableau Public.

All paths and logic are designed to be **fully reproducible** when this notebook is opened from GitHub in Google Colab.


In [1]:
# 1. Environment Setup – Libraries and Paths

import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("deep")

print("Environment OK – libraries loaded successfully.")

# Base directories inside Colab
clean_dir = "/content/data/cleaned"
tableau_dir = "/content/data/tableau"

os.makedirs(clean_dir, exist_ok=True)
os.makedirs(tableau_dir, exist_ok=True)

print("Data directories:")
print("  Cleaned data dir:", clean_dir)
print("  Tableau export dir:", tableau_dir)


Environment OK – libraries loaded successfully.
Data directories:
  Cleaned data dir: /content/data/cleaned
  Tableau export dir: /content/data/tableau


In [2]:
# 2. Retrieve Cleaned Dataset from GitHub

# RAW GitHub URL for the cleaned ZIP file
zip_url = "https://raw.githubusercontent.com/simasaadi/toronto-water-analytics/main/data/cleaned/toronto_aoc1_clean.zip"
zip_path = f"{clean_dir}/toronto_aoc1_clean.zip"

print("Downloading cleaned ZIP from GitHub...")
!wget -O "$zip_path" "$zip_url"

# Unzip into the cleaned directory
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(clean_dir)

print("\nFiles currently in cleaned folder:")
!ls -lh "$clean_dir"


--2025-11-23 22:05:00--  https://raw.githubusercontent.com/simasaadi/toronto-water-analytics/main/data/cleaned/toronto_aoc1_clean.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17649562 (17M) [application/zip]
Saving to: ‘/content/data/cleaned/toronto_aoc1_clean.zip’

/content/data/clean 100%[===================>]  16.83M  --.-KB/s    in 0.1s    

2025-11-23 22:05:00 (153 MB/s) - ‘/content/data/cleaned/toronto_aoc1_clean.zip’ saved [17649562/17649562]


Files currently in cleaned folder:
total 248M
-rw-r--r-- 1 root root 231M Nov 23 22:05 toronto_aoc1_clean.csv
-rw-r--r-- 1 root root  17M Nov 23 22:05 toronto_aoc1_clean.zip


In [3]:
# 3. Load Cleaned CSV into DataFrame

csv_path = f"{clean_dir}/toronto_aoc1_clean.csv"

df = pd.read_csv(csv_path, low_memory=False)

print("Dataset loaded successfully.")
print("Rows, Columns:", df.shape)

# Ensure activity_datetime is parsed as datetime
df["activity_datetime"] = pd.to_datetime(df["activity_datetime"], errors="coerce")

# Recompute basic temporal fields (safe even if they already exist)
df["year"] = df["activity_datetime"].dt.year
df["month"] = df["activity_datetime"].dt.month
df["dayofweek"] = df["activity_datetime"].dt.day_name()

df.head()


Dataset loaded successfully.
Rows, Columns: (770611, 22)


,id,doi,datasetname,monitoringlocationid,monitoringlocationname,monitoringlocationlatitude,monitoringlocationlongitude,monitoringlocationhorizontalcoordinatereferencesystem,monitoringlocationtype,activitytype,...,activitystarttime,samplecollectionequipmentname,characteristicname,resultvalue,resultunit,resultvaluetype,activity_datetime,year,month,dayofweek
0,119886859,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,...,0:00:00,Probe/Sensor,"Temperature, water",15.0,deg C,Actual,2019-07-25,2019,7,Thursday
1,119886893,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,...,0:00:00,Probe/Sensor,Conductivity,360.0,uS/cm,Actual,2021-07-15,2021,7,Thursday
2,119886896,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,...,0:00:00,NaN,pH,7.8,NaN,Actual,2019-07-18,2019,7,Thursday
3,119886916,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,...,0:00:00,Probe/Sensor,Conductivity,369.0,uS/cm,Actual,2020-09-03,2020,9,Thursday
4,119886936,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,...,0:00:00,Probe/Sensor,Conductivity,324.0,uS/cm,Actual,2020-09-17,2020,9,Thursday


## 4. Quick Sanity Check

Before exporting, I re-confirm:

- column data types  
- basic missingness  
- number of unique monitoring locations and characteristics  

This ensures the exports are based on a consistent, well-understood dataset.


In [4]:
print("\nColumn Types:")
print(df.dtypes)

print("\nMissing Values (count):")
print(df.isna().sum())

print("\nUnique Monitoring Locations:", df["monitoringlocationname"].nunique())
print("Unique Characteristics:", df["characteristicname"].nunique())



Column Types:
id                                                                int64
doi                                                              object
datasetname                                                      object
monitoringlocationid                                             object
monitoringlocationname                                           object
monitoringlocationlatitude                                      float64
monitoringlocationlongitude                                     float64
monitoringlocationhorizontalcoordinatereferencesystem            object
monitoringlocationtype                                           object
activitytype                                                     object
activitymedianame                                                object
activitystartdate                                                object
activitystarttime                                                object
samplecollectionequipmentname                    

## 5. Core Measurement Fact Table

The **fact table** is the main dataset that Tableau will use for flexible slicing and dicing.

It includes:

- time dimensions: `activity_datetime`, `year`, `month`, `dayofweek`  
- spatial dimensions: `monitoringlocationid`, `monitoringlocationname`, latitude, longitude  
- parameter dimension: `characteristicname`  
- measurement fields: `resultvalue`, `resultunit`, `resultvaluetype`  
- dataset metadata: `datasetname`, `doi`  

This table can be joined or filtered in multiple Tableau worksheets.


In [5]:
# 5. Build Core Fact Table

fact_cols = [
    "id",
    "doi",
    "datasetname",
    "activity_datetime",
    "year",
    "month",
    "dayofweek",
    "monitoringlocationid",
    "monitoringlocationname",
    "monitoringlocationlatitude",
    "monitoringlocationlongitude",
    "monitoringlocationhorizontalcoordinatereferencesystem",
    "monitoringlocationtype",
    "activitytype",
    "activitymedianame",
    "characteristicname",
    "resultvalue",
    "resultunit",
    "resultvaluetype",
]

fact_measurements = df[fact_cols].copy()

print("Fact table shape:", fact_measurements.shape)
fact_measurements.head()


Fact table shape: (770611, 19)


,id,doi,datasetname,activity_datetime,year,month,dayofweek,monitoringlocationid,monitoringlocationname,monitoringlocationlatitude,monitoringlocationlongitude,monitoringlocationhorizontalcoordinatereferencesystem,monitoringlocationtype,activitytype,activitymedianame,characteristicname,resultvalue,resultunit,resultvaluetype
0,119886859,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,2019-07-25,2019,7,Thursday,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,Surface Water,"Temperature, water",15.0,deg C,Actual
1,119886893,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,2021-07-15,2021,7,Thursday,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,Surface Water,Conductivity,360.0,uS/cm,Actual
2,119886896,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,2019-07-18,2019,7,Thursday,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,Surface Water,pH,7.8,NaN,Actual
3,119886916,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,2020-09-03,2020,9,Thursday,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,Surface Water,Conductivity,369.0,uS/cm,Actual
4,119886936,10.25976/zqwq-zh61,Swim Drink Fish Toronto Community Monitoring Hubs,2020-09-17,2020,9,Thursday,4544,Lake Ontario:Humber Bay West Park,43.61564,-79.47722,UNKWN,Lake/Pond,Field Msr/Obs,Surface Water,Conductivity,324.0,uS/cm,Actual


## 6. Trimmed Fact Table (Top 1% Removed)

To support **robust visualizations** that are less dominated by extreme outliers,  
I create a second fact table where I remove the **top 1%** of `resultvalue`:

- Compute the 99th percentile.  
- Filter out rows above this cutoff.  
- Keep the same structure as the main fact table.

This allows Tableau to use either the **full** or the **trimmed** dataset depending on the analysis.


In [6]:
metric_col = "resultvalue"

upper_1pct = df[metric_col].quantile(0.99)
df_trim = df[df[metric_col] <= upper_1pct].copy()

fact_measurements_trimmed = df_trim[fact_cols].copy()

print("99th percentile cutoff:", upper_1pct)
print("Full fact rows:    ", len(fact_measurements))
print("Trimmed fact rows: ", len(fact_measurements_trimmed))


99th percentile cutoff: 1820.0
Full fact rows:     770611
Trimmed fact rows:  762934


## 7. Monthly Overall Statistics

Here I compute **overall monthly statistics** for all measurements combined:

- resample by month using `activity_datetime`  
- compute `count`, `mean`, `median`, `min`, `max` of `resultvalue`  
- add `year`, numeric `month`, and month name (`month_name`) for easier use in Tableau

This table is ideal for high-level trend views (overall monthly behaviour).


In [7]:
# Ensure datetime index for resampling
df_time_indexed = df.set_index("activity_datetime").sort_index()

monthly_overall = (
    df_time_indexed
    .resample("ME")[metric_col]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

monthly_overall["year"] = monthly_overall["activity_datetime"].dt.year
monthly_overall["month"] = monthly_overall["activity_datetime"].dt.month
monthly_overall["month_name"] = monthly_overall["activity_datetime"].dt.month_name()

monthly_overall.head()


,activity_datetime,count,mean,median,min,max,year,month,month_name
0,1964-10-31,72,82.542500,9.50,0.01,964.0,1964,10,October
1,1964-11-30,76,402.848158,6.35,0.01,11634.0,1964,11,November
2,1964-12-31,87,149.158506,8.40,0.01,2778.0,1964,12,December
3,1965-01-31,108,646.370093,10.30,0.05,30654.0,1965,1,January
4,1965-02-28,58,170.560345,6.35,0.01,1790.0,1965,2,February


## 8. Monthly Statistics for Top Characteristics

To explore temporal behaviour of key parameters, I:

1. Identify the **top 6 most frequently measured characteristics**.  
2. Group by month and `characteristicname`.  
3. Compute `count`, `mean`, `median`, `min`, `max` for each combination.

This supports Tableau views comparing **trends across multiple parameters** on the same chart.


In [8]:
# Identify top 6 characteristics by count
top_chars = df["characteristicname"].value_counts().head(6).index

df_top = df[df["characteristicname"].isin(top_chars)].copy()
df_top_time = df_top.set_index("activity_datetime").sort_index()

monthly_char = (
    df_top_time
    .groupby([pd.Grouper(freq="ME"), "characteristicname"])[metric_col]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

monthly_char["year"] = monthly_char["activity_datetime"].dt.year
monthly_char["month"] = monthly_char["activity_datetime"].dt.month
monthly_char["month_name"] = monthly_char["activity_datetime"].dt.month_name()

print("Top characteristics used:")
print(top_chars.tolist())
monthly_char.head()


Top characteristics used:
['Total Phosphorus, mixed forms', 'Specific conductance', 'Chloride', 'Temperature, water', 'Kjeldahl nitrogen', 'Dissolved oxygen (DO)']


,activity_datetime,characteristicname,count,mean,median,min,max,year,month,month_name
0,1964-10-31,Dissolved oxygen (DO),10,8.1700,11.150,1.80,12.1,1964,10,October
1,1964-10-31,Kjeldahl nitrogen,4,1.1075,0.650,0.33,2.8,1964,10,October
2,1964-10-31,"Temperature, water",10,12.4500,11.000,9.50,18.0,1964,10,October
3,1964-10-31,"Total Phosphorus, mixed forms",4,5.1650,0.255,0.15,20.0,1964,10,October
4,1964-11-30,Chloride,1,360.0000,360.000,360.00,360.0,1964,11,November


## 9. Spatial Summary by Monitoring Location

For spatial analysis and map-based dashboards,  
I compute summary statistics at the **monitoring-location** level:

- group by: `monitoringlocationid`, `monitoringlocationname`, latitude, longitude  
- compute: `count`, `mean`, `median`, `min`, `max` of `resultvalue`

This table supports:

- choropleth-style maps (colour by mean/median)  
- bar charts ranking locations by typical levels  
- data-quality views showing sampling intensity (counts)


In [9]:
location_summary = (
    df.groupby(
        [
            "monitoringlocationid",
            "monitoringlocationname",
            "monitoringlocationlatitude",
            "monitoringlocationlongitude",
        ]
    )[metric_col]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

print("Location summary shape:", location_summary.shape)
location_summary.head()


Location summary shape: (648, 9)


,monitoringlocationid,monitoringlocationname,monitoringlocationlatitude,monitoringlocationlongitude,count,mean,median,min,max
0,08-179,Wilcox Lake,43.949049,-79.436034,33,123.665425,8.507056,0.000000,1795.490000
1,08-180,Eversley Lake,43.957772,-79.500823,21,9.174385,2.693537,0.000000,57.164785
2,08-183,Unnamed lake,43.747219,-79.735133,35,97.929927,5.300000,0.000000,1276.397274
3,08-184,Heart Lake,43.740539,-79.795426,32,75.534779,8.778776,-0.016255,785.380000
4,08-188,Fairy Lake,43.621242,-80.048136,35,79.210063,9.399859,0.000000,660.094137


## 10. Seasonal Median by Month

To capture **seasonal patterns**, I:

- group by numeric `month` across all years  
- compute the **median** of `resultvalue` per month  
- add a month name for readability

This table is perfect for a single **12-point seasonal line** or bar chart in Tableau.


In [10]:
seasonal_median = (
    df.groupby("month")[metric_col]
      .median()
      .reset_index()
      .rename(columns={metric_col: "median_resultvalue"})
)

seasonal_median["month_name"] = (
    pd.to_datetime(seasonal_median["month"], format="%m")
      .dt.month_name()
)

seasonal_median.sort_values("month", inplace=True)
seasonal_median.head()


,month,median_resultvalue,month_name
0,1,6.40,January
1,2,5.72,February
2,3,6.00,March
3,4,7.00,April
4,5,8.00,May


## 11. Save Export Tables for Tableau

All curated tables are now saved as `.csv` files in the  
`/content/data/tableau` directory:

- `fact_measurements_full.csv`  
- `fact_measurements_trimmed_top1pct_removed.csv`  
- `monthly_overall_stats.csv`  
- `monthly_top_characteristics_stats.csv`  
- `location_summary_stats.csv`  
- `seasonal_median_by_month.csv`  

These files can be downloaded from Colab and then:

1. Committed to the GitHub repository under `data/tableau/`  
2. Imported into Tableau Public as separate data sources.


In [11]:
# 11. Save All Exports to the Tableau Directory

fact_full_path = f"{tableau_dir}/fact_measurements_full.csv"
fact_trim_path = f"{tableau_dir}/fact_measurements_trimmed_top1pct_removed.csv"
monthly_overall_path = f"{tableau_dir}/monthly_overall_stats.csv"
monthly_char_path = f"{tableau_dir}/monthly_top_characteristics_stats.csv"
location_summary_path = f"{tableau_dir}/location_summary_stats.csv"
seasonal_median_path = f"{tableau_dir}/seasonal_median_by_month.csv"

fact_measurements.to_csv(fact_full_path, index=False)
fact_measurements_trimmed.to_csv(fact_trim_path, index=False)
monthly_overall.to_csv(monthly_overall_path, index=False)
monthly_char.to_csv(monthly_char_path, index=False)
location_summary.to_csv(location_summary_path, index=False)
seasonal_median.to_csv(seasonal_median_path, index=False)

print("Saved the following files in:", tableau_dir)
!ls -lh "$tableau_dir"


Saved the following files in: /content/data/tableau
total 413M
-rw-r--r-- 1 root root 208M Nov 23 22:08 fact_measurements_full.csv
-rw-r--r-- 1 root root 205M Nov 23 22:09 fact_measurements_trimmed_top1pct_removed.csv
-rw-r--r-- 1 root root  62K Nov 23 22:09 location_summary_stats.csv
-rw-r--r-- 1 root root  47K Nov 23 22:09 monthly_overall_stats.csv
-rw-r--r-- 1 root root 332K Nov 23 22:09 monthly_top_characteristics_stats.csv
-rw-r--r-- 1 root root  202 Nov 23 22:09 seasonal_median_by_month.csv


## 12. Quick Preview & Next Steps

Key export tables generated in this notebook include:

- `fact_measurements_full.csv` – raw measurement table  
- `fact_measurements_trimmed_top1pct_removed.csv` – version with top 1% outliers removed  
- `monthly_overall_stats.csv` – overall monthly trends  
- `monthly_top_characteristics_stats.csv` – monthly trends for top characteristics  
- `location_summary_stats.csv` – spatial summaries by monitoring location  
- `seasonal_median_by_month.csv` – seasonal pattern (median by month)

These files support downstream visualization and dashboard development in Tableau Public.

